<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN_02/blob/main/DBTT_04_mlp_ld_lq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sổ tay 04: Mô hình MLP nhận dạng Ld, Lq

**Dự án:** Nhận dạng tham số động cơ PMSM: mạng MLP nhận dạng Ld, Lq theo (id, iq); Rs tính đại số từ Ld, Lq đã nhận dạng.

**Nguyên tắc:** sổ tay chạy độc lập; mỗi lần chạy đều thiết lập lại môi trường từ đầu. Mã được viết, kiểm thử bằng dữ liệu mẫu, xuất thành tệp `.py` rồi đẩy lên GitHub theo bốn bước của PRD.

### Sơ đồ khối của sổ tay (BPMN 2.0, ISO/IEC 19510)



*Hình: Sơ đồ khối quá trình của sổ tay 04*

## Cell 1 – Thiết lập môi trường

In [1]:
# Cell 1 - Thiết lập môi trường từ đầu: gắn Drive, nhận mã truy cập, sao chép kho mã
import os  # Thao tác đường dẫn và tệp
import sys  # Điều chỉnh đường dẫn nạp mô-đun
import subprocess  # Chạy lệnh git và pip
from google.colab import drive, userdata  # Gắn Drive và đọc bí mật của Colab
drive.mount('/content/drive')  # Gắn Google Drive vào phiên Colab
GITHUB_USER = 'maivu18082005-hub'  # Tài khoản GitHub chứa kho mã
REPO_NAME = 'Parameter_Identification_of_PMSM_using_PINN_02'  # Tên kho mã
REPO_DIR = f'/content/{REPO_NAME}'  # Thư mục kho trên máy Colab
TOKEN = userdata.get('GITHUB_TOKEN')  # Mã truy cập lấy từ Colab Secrets
REMOTE = f'https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'  # Địa chỉ kho kèm mã truy cập


def run(cmd, check=True):  # Hàm chạy lệnh và che mã truy cập trong đầu ra
    r = subprocess.run(cmd, capture_output=True, text=True)  # Chạy lệnh, thu đầu ra
    out = (r.stdout + r.stderr).replace(TOKEN, '***')  # Thay mã truy cập bằng dấu sao
    if check and r.returncode != 0:  # Nếu lệnh lỗi và yêu cầu kiểm tra
        raise RuntimeError(out)  # Báo lỗi kèm đầu ra đã che
    return out  # Trả về đầu ra


def push_files(paths, message):  # Hàm đẩy tệp lên GitHub
    run(['git', '-C', REPO_DIR, 'config', 'user.name', GITHUB_USER])  # Đặt tên người commit
    run(['git', '-C', REPO_DIR, 'config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com'])  # Đặt email người commit
    run(['git', '-C', REPO_DIR, 'add'] + paths)  # Thêm các tệp vào vùng chờ
    if not run(['git', '-C', REPO_DIR, 'status', '--porcelain']).strip():  # Nếu không có thay đổi
        print('Không có thay đổi để đẩy')  # Thông báo
        return  # Kết thúc hàm
    run(['git', '-C', REPO_DIR, 'commit', '-m', message])  # Tạo commit
    run(['git', '-C', REPO_DIR, 'branch', '-M', 'main'])  # Đảm bảo nhánh là main
    run(['git', '-C', REPO_DIR, 'pull', '--rebase', REMOTE, 'main'], check=False)  # Đồng bộ trước khi đẩy
    print(run(['git', '-C', REPO_DIR, 'push', REMOTE, 'HEAD:main']))  # Đẩy lên GitHub


if os.path.isdir(os.path.join(REPO_DIR, '.git')):  # Nếu kho đã có trên máy
    run(['git', '-C', REPO_DIR, 'pull', REMOTE, 'main'], check=False)  # Cập nhật kho
else:  # Nếu chưa có
    run(['git', 'clone', REMOTE, REPO_DIR])  # Sao chép kho từ GitHub
sys.path.insert(0, REPO_DIR)  # Cho phép nạp mô-đun từ kho
TESTS_OK = False  # Cờ cho phép đẩy mã, chỉ bật sau khi kiểm thử đạt
print('Kho mã sẵn sàng tại', REPO_DIR)  # Thông báo hoàn tất

req = os.path.join(REPO_DIR, 'requirements.txt')  # Tệp yêu cầu thư viện
if os.path.exists(req):  # Nếu tệp tồn tại
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])  # Cài thư viện
import torch  # Thư viện học sâu
from src.utils.common import load_config, set_seed, get_paths  # Tiện ích dùng chung
cfg = load_config(os.path.join(REPO_DIR, 'configs', 'config.yaml'))  # Nạp cấu hình
set_seed(cfg['project']['seed'])  # Cố định hạt giống
PATHS = get_paths(cfg)  # Đường dẫn tuyệt đối trên Drive
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Chọn thiết bị tính toán
print('Thiết bị:', DEVICE, '| Torch:', torch.__version__)  # In thông tin môi trường

Mounted at /content/drive
Kho mã sẵn sàng tại /content/Parameter_Identification_of_PMSM_using_PINN_02
Thiết bị: cpu | Torch: 2.14.1+cpu


## Cell 2 – Nạp thống kê nhãn
Khoảng đầu ra của mạng lấy từ nhãn Ld, Lq của tập huấn luyện mở rộng theo biên; hệ số chuẩn hóa đầu vào là dòng đỉnh định mức.

In [2]:
# Cell 2 - Nạp thống kê nhãn do sổ tay 02 tạo ra
import json  # Đọc thống kê JSON
import numpy as np  # Thư viện số học mảng
p = os.path.join(PATHS['processed'], 'stats.json')  # Đường dẫn thống kê nhãn
assert os.path.exists(p), 'Chạy sổ tay 02 trước để tạo stats.json'  # Dừng nếu thiếu
with open(p, encoding='utf-8') as fh:  # Mở tệp thống kê
    stats = json.load(fh)  # Đọc thống kê
print({k: (round(stats[k]['lo'], 6), round(stats[k]['hi'], 6)) for k in ('Ld', 'Lq')})  # In khoảng đầu ra của mạng

{'Ld': (0.0009, 0.001598), 'Lq': (0.001013, 0.001726)}


## Cell 3 – Viết `src/models/ld_lq_mlp.py`
Mạng MLP hai đầu vào (id, iq), hai đầu ra (Ld, Lq); chuẩn hóa đầu vào cố định, khởi tạo Xavier, đầu ra bị chặn bằng hàm sigmoid trong khoảng vật lý.

In [3]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/src/models/ld_lq_mlp.py
"""Mạng MLP: ánh xạ (id, iq) sang (Ld, Lq), chỉ học từ nhãn."""  # Mô tả mô-đun
import torch  # Thư viện học sâu
import torch.nn as nn  # Các lớp mạng nơ-ron

ACTIVATIONS = {"tanh": nn.Tanh, "relu": nn.ReLU, "silu": nn.SiLU}  # Bảng hàm kích hoạt


class LdLqMLP(nn.Module):  # Lớp mạng MLP
    """MLP với chuẩn hóa đầu vào cố định và đầu ra bị chặn trong khoảng vật lý."""  # Mô tả lớp

    def __init__(self, in_scale, out_lo, out_hi, hidden_sizes=(32, 32), activation="tanh"):  # Hàm khởi tạo
        """Dựng mạng với hệ số chuẩn hóa đầu vào và cận đầu ra."""  # Mô tả hàm
        super().__init__()  # Gọi lớp cha
        self.hidden_sizes, self.activation = list(hidden_sizes), activation  # Lưu kiến trúc
        self.register_buffer("in_scale", torch.tensor(in_scale, dtype=torch.float32))  # Hệ số chia đầu vào
        self.register_buffer("out_lo", torch.tensor(out_lo, dtype=torch.float32))  # Cận dưới đầu ra
        self.register_buffer("out_hi", torch.tensor(out_hi, dtype=torch.float32))  # Cận trên đầu ra
        sizes = [len(in_scale)] + self.hidden_sizes + [len(out_lo)]  # Danh sách kích thước các lớp
        layers = []  # Danh sách lớp
        for i in range(len(sizes) - 1):  # Duyệt từng cặp lớp liên tiếp
            layers.append(nn.Linear(sizes[i], sizes[i + 1]))  # Thêm lớp tuyến tính
            if i < len(sizes) - 2:  # Nếu chưa phải lớp cuối
                layers.append(ACTIVATIONS[activation]())  # Thêm hàm kích hoạt
        self.net = nn.Sequential(*layers)  # Gộp thành mạng tuần tự
        self.init_weights()  # Khởi tạo trọng số

    def init_weights(self):  # Khởi tạo trọng số
        """Khởi tạo Xavier cho trọng số và bằng 0 cho độ lệch."""  # Mô tả hàm
        for m in self.net:  # Duyệt từng lớp
            if isinstance(m, nn.Linear):  # Chỉ xử lý lớp tuyến tính
                nn.init.xavier_uniform_(m.weight)  # Khởi tạo Xavier
                nn.init.zeros_(m.bias)  # Độ lệch bằng 0

    def forward(self, x):  # Lan truyền thuận
        """Nhận (id, iq) theo ampe, kích thước (N,2); trả về (Ld, Lq) theo henry, kích thước (N,2)."""  # Mô tả hàm
        z = self.net(x / self.in_scale)  # Chuẩn hóa rồi qua mạng
        return self.out_lo + (self.out_hi - self.out_lo) * torch.sigmoid(z)  # Ánh xạ vào khoảng [lo, hi]

    def count_parameters(self):  # Đếm tham số
        """Trả về tổng số tham số huấn luyện được."""  # Mô tả hàm
        return sum(p.numel() for p in self.parameters() if p.requires_grad)  # Cộng số phần tử

    def get_config(self):  # Lấy cấu hình
        """Trả về từ điển đủ để dựng lại mạng."""  # Mô tả hàm
        return {"in_scale": self.in_scale.tolist(), "out_lo": self.out_lo.tolist(), "out_hi": self.out_hi.tolist(), "hidden_sizes": self.hidden_sizes, "activation": self.activation}  # Gói cấu hình


def build_mlp(cfg, stats):  # Hàm dựng mạng từ cấu hình
    """Dựng LdLqMLP từ cấu hình và thống kê nhãn Ld, Lq của tập huấn luyện."""  # Mô tả hàm
    i_base = cfg["motor"]["i_rated_peak"]  # Dòng cơ sở (A)
    in_scale = [i_base, i_base]  # Hệ số chuẩn hóa (id, iq)
    lo = [stats[k]["lo"] for k in ("Ld", "Lq")]  # Cận dưới đầu ra
    hi = [stats[k]["hi"] for k in ("Ld", "Lq")]  # Cận trên đầu ra
    return LdLqMLP(in_scale, lo, hi, cfg["model"]["hidden_sizes"], cfg["model"]["activation"])  # Dựng mạng

Writing /content/Parameter_Identification_of_PMSM_using_PINN_02/src/models/ld_lq_mlp.py


## Cell 4 – Kiểm thử đơn vị
Kiểm tra hình dạng, cận đầu ra, tính xác định, số tham số, dòng gradient, nạp lại trọng số và việc mạng từ chối đầu vào sai số cột.

In [4]:
# Cell 4 - Kiểm thử đơn vị cho ld_lq_mlp.py
import importlib  # Nạp lại mô-đun vừa ghi
import src.models.ld_lq_mlp as lm  # Mô-đun cần kiểm thử
importlib.reload(lm)  # Dùng bản mới ghi
m = lm.build_mlp(cfg, stats)  # Dựng mạng từ cấu hình
x = torch.rand(256, 2) * 130.0 - 65.0  # Đầu vào ngẫu nhiên (id, iq) trong dải ±65 A
y = m(x)  # Lan truyền thuận
assert y.shape == (256, 2), 'Sai hình dạng đầu ra'  # Kiểm tra hình dạng
assert (y >= m.out_lo).all() and (y <= m.out_hi).all(), 'Đầu ra vượt cận'  # Kiểm tra cận đầu ra
assert torch.allclose(m.out_lo, torch.tensor([stats['Ld']['lo'], stats['Lq']['lo']], dtype=torch.float32)), 'Cận dưới phải lấy từ nhãn Ld, Lq'  # Kiểm tra nguồn cận
assert torch.allclose(m.in_scale, torch.full((2,), cfg['motor']['i_rated_peak'])), 'Hệ số chuẩn hóa phải là dòng đỉnh định mức'  # Kiểm tra chuẩn hóa
m.eval()  # Chuyển sang chế độ suy luận
assert torch.equal(m(x), m(x)), 'Lan truyền thuận phải xác định'  # Kiểm tra tính xác định
sizes = [2] + cfg['model']['hidden_sizes'] + [2]  # Kích thước các lớp
expect = sum(sizes[i] * sizes[i + 1] + sizes[i + 1] for i in range(len(sizes) - 1))  # Số tham số kỳ vọng
assert m.count_parameters() == expect, 'Sai số tham số'  # Kiểm tra số tham số
assert not any(n in ('in_scale', 'out_lo', 'out_hi') for n, _ in m.named_parameters()), 'Hằng số chuẩn hóa không được huấn luyện'  # Kiểm tra hằng số
m.zero_grad()  # Xóa gradient
m(x).sum().backward()  # Lan truyền ngược
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in m.parameters()), 'Gradient không hợp lệ'  # Kiểm tra gradient
m2 = lm.build_mlp(cfg, stats)  # Dựng mạng thứ hai
m2.load_state_dict(m.state_dict())  # Nạp trọng số
m2.eval()  # Chuyển sang chế độ suy luận
assert torch.equal(m(x), m2(x)), 'Nạp lại trọng số phải cho cùng kết quả'  # Kiểm tra nạp lại
m3 = lm.LdLqMLP(**m.get_config())  # Dựng lại từ cấu hình đã lưu
m3.load_state_dict(m.state_dict())  # Nạp trọng số
assert m3.get_config() == m.get_config(), 'get_config phải đủ để dựng lại mạng'  # Kiểm tra cấu hình
try:  # Mạng chỉ nhận hai cột (id, iq)
    m(torch.rand(4, 5))  # Đưa vào năm cột
    raise AssertionError('Đầu vào sai số cột phải báo lỗi')  # Không được chạy tới đây
except RuntimeError:  # Lỗi mong đợi
    pass  # Đúng như dự kiến
mac = sum(sizes[i] * sizes[i + 1] for i in range(len(sizes) - 1))  # Số phép nhân-cộng mỗi lần suy luận
print('Số tham số:', m.count_parameters(), '| Flash trọng số (float32):', 4 * m.count_parameters(), 'byte | MAC:', mac)  # In chi phí tính toán
TESTS_OK = True  # Bật cờ cho phép đẩy mã
print('Toàn bộ kiểm thử đơn vị đạt')  # Thông báo

Số tham số: 1218 | Flash trọng số (float32): 4872 byte | MAC: 1152
Toàn bộ kiểm thử đơn vị đạt


## Cell 5 – Bước 4: đẩy lên GitHub

In [5]:
# Cell cuối - Bước 4: đẩy tệp đã kiểm thử đạt lên GitHub
assert TESTS_OK, 'Chưa đạt toàn bộ kiểm thử, không được đẩy mã'  # Chặn đẩy mã khi kiểm thử chưa đạt
push_files(['src/models/ld_lq_mlp.py'], 'Thêm ld_lq_mlp.py')  # Đẩy các tệp lên nhánh main

To https://github.com/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN_02.git
   e125dcb..829395a  HEAD -> main

